# Plot AOI mean AGB

### Import required packages

In [1]:
import os
import geopandas as gpd
import snowflake.connector

### Connect to the database and fetch the AOI data
- Note that the environment variables need to be set with the database credentials. 
- Replace 'my-aoi-id' with your AOI ID. 
- For Planet, dividing by 0.476 transforms from aboveground biomass carbon to total aboveground biomass.

In [2]:
with snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
        database=os.environ["SNOWFLAKE_DATABASE"],
        user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
) as conn:
    
    cur = conn.cursor().execute('''
        select
            st_aswkt(boundary) as boundary,
            (c.aboveground_biomass_stock + k.living_aboveground_biomass + (p.aboveground_live_carbon_density / 0.476)) / 3 as mean_agb
        from chloris.aboveground_biomass_stock_and_change c
        join kanop.standard k using (aoi_id, year, x, y)
        join planet.forest_carbon_diligence p using (aoi_id, year, x, y)
        where 
            aoi_id = 'my-aoi-id' and 
            year = 2022
    ''')

    df = cur.fetch_pandas_all()

### Plot AOI
Plot the mean AGB of all three providers in 2022 across the entire AOI.

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(
    df["BOUNDARY"],
    crs="EPSG:4326",
))

gdf.plot(column="MEAN_AGB")